In [6]:
# this file depends on many definitions found in configuration_eval
%run configuration_eval.ipynb

import json
import datetime

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv("vars.env")

client = Anthropic()
SYSTEM_PROMPT = open("whisper_continuation_prompt.md").read()

#performs worse
#SYSTEM_PROMPT = open("whisper_terminology_prompt.md").read()

def run_prompt_engineering_eval(transcript_model, model, mp3_file, reference_srt):

    diagnostics_dir = os.path.join("../outputs/prompt_engineering_outputs", datetime.datetime.now().strftime("%H:%M:%S"))
    os.makedirs(diagnostics_dir, exist_ok=True)
    file_prefix = os.path.splitext(os.path.basename(mp3_file))[0][:5]

    srt1, transcription1 = whisper_transcribe(mp3_file, transcript_model, "", False)

    with open(os.path.join(diagnostics_dir, file_prefix + "transcription1.txt"), "w", encoding="utf-8") as f:
        f.write(transcription1)

    start_srt = os.path.join("../outputs/prompt_engineering_outputs", "srt1.srt")
    with open(start_srt, "w", encoding="utf-8") as f:
        f.write(srt1)

    resp = client.messages.create(
        model=model,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": transcription1},
        ],
    )

    # skip thinking blocks
    whisper_prompt = next((block.text for block in resp.content if block.type == "text"), None)

    with open(os.path.join(diagnostics_dir, file_prefix + "whisper_prompt.txt"), "w", encoding="utf-8") as f:
        f.write(whisper_prompt)

    #finally the step where we get a transcription using our conditioning prompt generated at runtime!
    srt2, transcription2 = whisper_transcribe(mp3_file, transcript_model, whisper_prompt, False)

    with open(os.path.join(diagnostics_dir, file_prefix + "transcription2.txt"), "w", encoding="utf-8") as f:
        f.write(transcription2)

    end_srt = os.path.join("../outputs/prompt_engineering_outputs", "srt2.srt")
    with open(end_srt, "w", encoding="utf-8") as f:
        f.write(srt2)

    #calculate_wer from configuration_eval.ipynb
    score1 = calculate_wer(reference_srt, start_srt)
    score2 = calculate_wer(reference_srt, end_srt)
    with open(os.path.join(diagnostics_dir, file_prefix + "scores.txt"), "w", encoding="utf-8") as f:
        f.write(f"{score1}, {score2}")

    return score1, score2

def multi_prompt_engineering_eval(transcript_model, model, eval_path):
    scores1 = {}
    scores2 = {}

    for entry in sorted(os.listdir(eval_path)):
        folder = os.path.join(eval_path, entry)
        if not os.path.isdir(folder):
            continue

        reference_srt = os.path.join(folder, "corrected_transcript.srt")
        if not os.path.exists(reference_srt):
            continue

        mp3_files = glob.glob(os.path.join(folder, "*.mp3"))
        if not mp3_files:
            continue
        mp3_file = mp3_files[0]

        scores1[entry], scores2[entry] = run_prompt_engineering_eval(transcript_model, model, mp3_file, reference_srt)
        
    mean_errorscore1 = sum(scores1.values()) / len(scores1) if scores1 else 0.0
    mean_errorscore2 = sum(scores2.values()) / len(scores2) if scores2 else 0.0
    return scores1, mean_errorscore1, scores2, mean_errorscore2


In [7]:
transcript_model = "whisper-large-v3-turbo"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

Average before score: 0.02499939628287887
Average after score: 0.03506278353751247
After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.


In [8]:
transcript_model = "whisper-large-v3"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

Average before score: 0.025637735459090174
Average after score: 0.08953063528876715
After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.


In [9]:
transcript_model = "whisper-1"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

Average before score: 0.020708627838201585
Average after score: 0.01624707836759883
After auto prompt engineering, transcript accuracy improved.
